# Prepare an MD system with mdprep

This notebook **runs** a manifest. It does not build one: the
[mdprep input builder](https://miqueleg.github.io/mdprep/) does that in your
browser, showing you the structure while you answer, and hands you a bundle
containing the structure and the manifest together.

If you would rather answer the questions here, cell 2b does the same thing on
the command line.

Order: upload or fetch your inputs, install the toolchain the manifest needs,
prepare the system, then optionally equilibrate it.


## 1. Install mdprep

This is a small pure-Python install. The heavy external programs come later,
once the manifest has said which of them this job actually uses.


In [ ]:
%pip install -q "mdprep @ git+https://github.com/miqueleg/mdprep@master"
import mdprep
print("mdprep", mdprep.__version__)


## 2. Load your inputs

Run **2a** if you have a bundle from the web builder. Run **2b** instead to
answer the questions here.


### 2a. Upload a bundle from the web builder

In [ ]:
import pathlib, zipfile
from google.colab import files

WORKDIR = pathlib.Path("/content/mdprep")
WORKDIR.mkdir(parents=True, exist_ok=True)
%cd {WORKDIR}

uploaded = files.upload()           # choose mdprep-bundle.zip
zipfile.ZipFile(next(iter(uploaded))).extractall(WORKDIR)
MANIFEST = WORKDIR / "system.yaml"
print(sorted(p.name for p in WORKDIR.iterdir()))


### 2b. Or answer the questions here

`mdprep plan` lists only the decisions your structure actually needs, and
`mdprep init --interactive` asks them. Upload a PDB first, or fetch one by ID.


In [ ]:
import pathlib, urllib.request

WORKDIR = pathlib.Path("/content/mdprep")
WORKDIR.mkdir(parents=True, exist_ok=True)
%cd {WORKDIR}

PDB_ID = "6NIE"  #@param {type:"string"}
structure = WORKDIR / f"{PDB_ID}.pdb"
if not structure.exists():
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.pdb", structure)

!mdprep plan {structure}


In [ ]:
MANIFEST = WORKDIR / "system.yaml"
!mdprep init {structure} --interactive -o {MANIFEST} --overwrite


## 3. Install the external toolchain

mdprep shells out to `tleap` and friends, which pip cannot provide but
conda-forge can. Only the programs this manifest will actually invoke are
installed, so a plain protein-ligand job does not wait for a quantum chemistry
package it never calls.

micromamba is used rather than condacolab because it installs into its own
prefix instead of replacing this notebook's Python, so the kernel does not have
to restart.

This is a large download and takes several minutes, but only once per session.


In [ ]:
import os, subprocess, yaml
from mdprep.workflows.runtime import conda_packages_for, micromamba_setup_commands

manifest = yaml.safe_load(MANIFEST.read_text())
packages = conda_packages_for(manifest)
print("this manifest needs:", ", ".join(packages))

script = "\n".join(
    ["set -eux"]
    + micromamba_setup_commands(
        manifest,
        mdprep_spec='"mdprep @ git+https://github.com/miqueleg/mdprep@master"',
    )
)
result = subprocess.run(["bash", "-c", script], text=True)
if result.returncode:
    raise RuntimeError("toolchain install failed; see the output above")

RUNTIME = ["/content/bin/micromamba", "run", "-n", "mdprep-runtime"]
ENV = {**os.environ, "MAMBA_ROOT_PREFIX": "/content/mamba-root"}
print("ready")


## 4. Prepare the system

Output streams into this cell. Everything mdprep runs is recorded under the
manifest's `output_dir`, including every external command and its logs.


In [ ]:
result = subprocess.run(
    [*RUNTIME, "mdprep", "prepare", str(MANIFEST), "--overwrite"],
    cwd=WORKDIR, env=ENV, text=True,
)
if result.returncode:
    raise RuntimeError(f"mdprep prepare failed with exit code {result.returncode}")
print("prepared")


## 5. Check what was built

`mdprep validate` re-reads the finished topology and coordinates rather than
trusting the build.


In [ ]:
import glob
prmtop = sorted(glob.glob(str(WORKDIR / "**" / "final" / "system.prmtop"), recursive=True))
inpcrd = sorted(glob.glob(str(WORKDIR / "**" / "final" / "system.inpcrd"), recursive=True))
print("topology:", prmtop, "\ncoordinates:", inpcrd)

if prmtop and inpcrd:
    subprocess.run([*RUNTIME, "mdprep", "validate", prmtop[0], inpcrd[0]],
                   cwd=WORKDIR, env=ENV, text=True, check=False)


## 6. Optional: equilibrate and run

Only if `molecular_dynamics.enabled` is true in the manifest. Select a GPU
runtime first (Runtime -> Change runtime type), or this will be very slow.

A free Colab session is time limited. mdprep writes checkpoints throughout, so
a disconnected run can be resumed rather than restarted.


In [ ]:
if manifest.get("molecular_dynamics", {}).get("enabled"):
    subprocess.run([*RUNTIME, "mdprep", "run-md", str(MANIFEST)],
                   cwd=WORKDIR, env=ENV, text=True, check=False)
else:
    print("molecular_dynamics.enabled is false in this manifest; nothing to run.")


## 7. Download the results

In [ ]:
import shutil
archive = shutil.make_archive("/content/mdprep-output", "zip", root_dir=WORKDIR)
print(archive)
files.download(archive)
